# Graph RAG con Cypher — COSORA (Neo4j)

Notebook **autocontenido** para Colab/local.

**Pipeline:** JSON → Neo4j → Cypher → chunks → respuesta LLM.

**Evaluación (sec. 7):** calidad de **respuesta** (LLM-as-Judge) + checks **deterministas** (citas, overlap de chunks). **Sec. 8:** pruebas iterativas.

**Prerequisitos:** JSON batch1 en Drive, Chroma e5, Neo4j Aura + `OPENAI_API_KEY` en `.env` (sec. 0.1).


## 0. Setup


In [11]:
%pip install -q neo4j chromadb sentence-transformers rank_bm25 python-dotenv pandas nltk openai

import nltk
nltk.download("punkt", quiet=True)
nltk.download("snowball_data", quiet=True)


True

### 0.1 Neo4j Aura (una sola vez)

1. Crear instancia gratuita en [Neo4j Aura Free](https://neo4j.com/cloud/platform/aura-graph-database/)
2. Copiar **URI**, **usuario** y **contraseña**
3. Añadir al `.env` del Drive (`MyDrive/variablentorno/.env`) o al `.env` local:

```
NEO4J_URI=neo4j+s://xxxx.databases.neo4j.io
NEO4J_USER=neo4j
NEO4J_PASSWORD=...
NEO4J_DATABASE=neo4j
```

Ver también [`.env.example`](../../.env.example) en el repo.


In [12]:
import json
import os
import re
import sys
import unicodedata
from collections import Counter
from pathlib import Path
from typing import Any

import chromadb
import numpy as np
import pandas as pd
from neo4j import GraphDatabase
from openai import OpenAI
from rank_bm25 import BM25Okapi

IN_COLAB = "google.colab" in sys.modules
RUNTIME = "colab" if IN_COLAB else "local"

# ─── Config ───────────────────────────────────────────────────────────
EMBED_BACKEND = "e5"
GRAPH_JSON_NAME = "graph_actas_e5_original_sub10_bal_raw.json"
SOURCE_BATCH = "sub10"
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")
WIPE_BEFORE_LOAD = True

RETRIEVAL_K = 100
TOP_N = 10
RRF_K = 60
MATCH_MIN_OVERLAP = 0.5
MATCH_STRATEGY = "wordset"
GEN_MODEL = "gpt-4o-mini"
LLM_JUDGE_MODEL = "gpt-4o-mini"
ANSWER_EVAL_ENABLED = True

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

from dotenv import load_dotenv
if RUNTIME == "colab":
    load_dotenv("/content/drive/MyDrive/variablentorno/.env")
elif Path(".env").exists():
    load_dotenv(".env")
elif Path("../../.env").exists():
    load_dotenv("../../.env")

NEO4J_URI = os.getenv("NEO4J_URI", "")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    print("⚠️  OPENAI_API_KEY no encontrada")
if not NEO4J_URI or not NEO4J_PASSWORD:
    print("⚠️  NEO4J_URI / NEO4J_PASSWORD no configurados — sec. 3+ fallará hasta que los añadas al .env")

client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None


def resolve_paths(runtime: str):
    if runtime == "colab":
        docs_dir = "/content/drive/MyDrive/RAG_UPC_Final_project"
        chroma_path = f"{docs_dir}/chroma_db"
        graph_dir = f"{docs_dir}/graph"
    else:
        nb_dir = Path(".").resolve()
        root = nb_dir.parents[1] if nb_dir.name == "experiments" else (
            nb_dir.parent if nb_dir.name == "notebooks" else nb_dir
        )
        docs_dir = str(root / "data" / "raw")
        chroma_path = str(root / "data" / "chroma_db")
        graph_dir = str(root / "data" / "graph")
    Path(graph_dir).mkdir(parents=True, exist_ok=True)
    return docs_dir, chroma_path, graph_dir


DOCS_DIR, CHROMA_PATH, GRAPH_DIR = resolve_paths(RUNTIME)
GRAPH_JSON_PATH = Path(GRAPH_DIR) / GRAPH_JSON_NAME

print(f"RUNTIME={RUNTIME}")
print(f"GRAPH_DIR={GRAPH_DIR}")
print(f"GRAPH_JSON={GRAPH_JSON_PATH}  exists={GRAPH_JSON_PATH.exists()}")
print(f"CHROMA_PATH={CHROMA_PATH}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RUNTIME=colab
GRAPH_DIR=/content/drive/MyDrive/RAG_UPC_Final_project/graph
GRAPH_JSON=/content/drive/MyDrive/RAG_UPC_Final_project/graph/graph_actas_e5_original_sub10_bal_raw.json  exists=True
CHROMA_PATH=/content/drive/MyDrive/RAG_UPC_Final_project/chroma_db


## 1. Inspección del JSON


In [13]:
def load_graph_json(path: Path | None = None) -> dict:
    path = path or GRAPH_JSON_PATH
    if not path.exists():
        hits = sorted(Path(GRAPH_DIR).glob("graph_actas_e5_original_*.json"))
        if hits:
            path = hits[-1]
            print(f"⚠️  Fallback: {path.name}")
        else:
            raise FileNotFoundError(
                f"No hay JSON en {GRAPH_DIR}. "
                "Ejecuta graph_rag.ipynb sec. 2 o copia el archivo desde Drive."
            )
    with open(path, encoding="utf-8") as f:
        gd = json.load(f)
    gd["_path"] = str(path)
    return gd


def print_graph_stats(gd: dict):
    rels = [tuple(r) for r in gd["relations"]]
    nodes = set()
    for s, _, o in rels:
        nodes.add(s)
        nodes.add(o)
    preds = Counter(r for _, r, _ in rels)
    meta = gd.get("meta", {})
    print(f"Archivo: {gd.get('_path', '?')}")
    print(f"  Nodos únicos:     {len(nodes)}")
    print(f"  Triples:          {len(rels)}")
    print(f"  Tipos relación:   {len(preds)}")
    print(f"  Top relaciones:   {preds.most_common(5)}")
    print(f"  meta.kg_source:   {meta.get('kg_source')}")
    print(f"  meta.subset_size: {meta.get('subset_size')}")
    sources = meta.get("sources") or []
    print(f"  Actas (sources):  {len(sources)}")
    if sources:
        print(f"    ej: {sources[:3]} ...")


graph_dict = load_graph_json()
print_graph_stats(graph_dict)
batch_sources = set(graph_dict.get("meta", {}).get("sources") or [])


Archivo: /content/drive/MyDrive/RAG_UPC_Final_project/graph/graph_actas_e5_original_sub10_bal_raw.json
  Nodos únicos:     508
  Triples:          426
  Tipos relación:   270
  Top relaciones:   [('solicita', 13), ('informa que', 11), ('es parte de', 11), ('en', 10), ('is part of', 8)]
  meta.kg_source:   original
  meta.subset_size: 10
  Actas (sources):  10
    ej: ['244170-DOB-AVO-05-V00-A0', '252562-DO-AVO-02-V01', '254275-DO-AVO-14-V07'] ...


## 2. Esquema Neo4j


In [14]:
def norm_entity(name: str) -> str:
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", name.lower().strip())


SCHEMA_CONSTRAINTS = [
    "CREATE CONSTRAINT entity_norm IF NOT EXISTS FOR (e:Entity) REQUIRE e.norm IS UNIQUE",
]

print("Esquema v1:")
print("  (:Entity {name, norm, batch})-[:RELATED {predicate, triple_idx, batch}]->(:Entity)")


Esquema v1:
  (:Entity {name, norm, batch})-[:RELATED {predicate, triple_idx, batch}]->(:Entity)


## 3. Carga JSON → Neo4j


In [15]:
def get_neo4j_driver():
    if not NEO4J_URI or not NEO4J_PASSWORD:
        raise ValueError("Configura NEO4J_URI y NEO4J_PASSWORD en .env")
    drv = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    drv.verify_connectivity()
    print("✅ Neo4j conectado")
    return drv


def setup_schema(driver):
    with driver.session(database=NEO4J_DATABASE) as session:
        for q in SCHEMA_CONSTRAINTS:
            session.run(q)
    print("✅ Constraints aplicados")


def wipe_batch(driver, batch: str = SOURCE_BATCH):
    with driver.session(database=NEO4J_DATABASE) as session:
        session.run(
            "MATCH (n) WHERE n.batch = $batch DETACH DELETE n",
            batch=batch,
        )
        if WIPE_BEFORE_LOAD:
            session.run("MATCH (n) DETACH DELETE n")
    print(f"✅ Grafo limpiado (WIPE_BEFORE_LOAD={WIPE_BEFORE_LOAD})")


def build_load_rows(rels: list[tuple], batch: str = SOURCE_BATCH) -> list[dict]:
    rows = []
    for idx, (s, p, o) in enumerate(rels):
        rows.append({
            "triple_idx": idx,
            "subject": s,
            "s_norm": norm_entity(s),
            "predicate": p,
            "object": o,
            "o_norm": norm_entity(o),
            "batch": batch,
        })
    return rows


LOAD_CYPHER = """
UNWIND $rows AS row
MERGE (s:Entity {norm: row.s_norm})
  ON CREATE SET s.name = row.subject, s.batch = row.batch
  ON MATCH SET s.name = coalesce(s.name, row.subject)
MERGE (o:Entity {norm: row.o_norm})
  ON CREATE SET o.name = row.object, o.batch = row.batch
  ON MATCH SET o.name = coalesce(o.name, row.object)
MERGE (s)-[r:RELATED {triple_idx: row.triple_idx, batch: row.batch}]->(o)
  SET r.predicate = row.predicate
"""


def load_graph_to_neo4j(driver, gd: dict, batch_size: int = 100):
    rels = [tuple(r) for r in gd["relations"]]
    rows = build_load_rows(rels)
    with driver.session(database=NEO4J_DATABASE) as session:
        for i in range(0, len(rows), batch_size):
            batch = rows[i : i + batch_size]
            session.run(LOAD_CYPHER, rows=batch)
            print(f"  cargados {min(i + batch_size, len(rows))}/{len(rows)} triples")
    print(f"✅ {len(rows)} triples cargados en Neo4j")


def verify_neo4j(driver):
    with driver.session(database=NEO4J_DATABASE) as session:
        n_ent = session.run("MATCH (e:Entity) RETURN count(e) AS c").single()["c"]
        n_rel = session.run("MATCH ()-[r:RELATED]->() RETURN count(r) AS c").single()["c"]
        sample = session.run(
            "MATCH (a:Entity)-[r:RELATED]->(b:Entity) "
            "RETURN a.name, r.predicate, b.name LIMIT 5"
        ).data()
    print(f"Neo4j: {n_ent} entidades, {n_rel} relaciones RELATED")
    display(pd.DataFrame(sample))


driver = get_neo4j_driver()
setup_schema(driver)
wipe_batch(driver)
load_graph_to_neo4j(driver, graph_dict)
verify_neo4j(driver)


✅ Neo4j conectado
✅ Constraints aplicados
✅ Grafo limpiado (WIPE_BEFORE_LOAD=True)
  cargados 100/426 triples
  cargados 200/426 triples
  cargados 300/426 triples
  cargados 400/426 triples
  cargados 426/426 triples
✅ 426 triples cargados en Neo4j
Neo4j: 494 entidades, 426 relaciones RELATED


,a.name,r.predicate,b.name
0,marquesina,se encuentra en,vía 02
1,marquesina,de la,vía 02
2,marquesina,es parte de,andén de vía 02
3,Compañía eléctrica,debe ser informada por la,constructora
4,constructora,informa que,trabajos


## 4. Chroma baseline (dense + BM25)


In [16]:
EMBED_BACKENDS: dict[str, dict[str, Any]] = {
    "e5": {
        "model_id": "intfloat/multilingual-e5-base",
        "collection": "cosora_chunks_e5",
        "type": "sentence_transformers",
        "query_prefix": "query: ",
        "doc_prefix": "passage: ",
    },
}

TECH_TOKEN_PATTERN = re.compile(
    r"^(AR-\d+|UTE|DF|DEO|DO|PPI|[A-Z]{2,}\d*|\d+[A-Z-]+)$",
    re.IGNORECASE,
)

STOPWORDS_ES = {
    "de", "la", "que", "el", "en", "y", "a", "los", "del", "se", "las", "por", "un",
    "para", "con", "no", "una", "su", "al", "es", "lo", "como", "más", "pero", "sus",
    "le", "ya", "o", "este", "sí", "porque", "esta", "entre", "cuando", "muy", "sin",
    "sobre", "también", "me", "hasta", "hay", "donde", "quien", "desde", "todo", "nos",
    "durante", "todos", "uno", "les", "ni", "contra", "otros", "ese", "eso", "ante",
}


def backend_config(backend):
    return EMBED_BACKENDS[backend]


class Embedder:
    def __init__(self, backend):
        self.backend = backend
        self.cfg = backend_config(backend)
        self._st_model = None

    def _load(self):
        if self._st_model is None:
            from sentence_transformers import SentenceTransformer
            self._st_model = SentenceTransformer(self.cfg["model_id"])

    def embed_one(self, text, *, is_query=False):
        self._load()
        prefix = self.cfg["query_prefix"] if is_query else self.cfg["doc_prefix"]
        vec = self._st_model.encode(prefix + text)
        return vec.tolist() if hasattr(vec, "tolist") else list(vec)

    def embed_batch(self, texts, *, is_query=False, batch_size=32):
        self._load()
        prefix = self.cfg["query_prefix"] if is_query else self.cfg["doc_prefix"]
        vecs = self._st_model.encode([prefix + t for t in texts], batch_size=batch_size, show_progress_bar=False)
        return vecs.tolist()


def strip_doc_prefix(text, backend):
    prefix = backend_config(backend).get("doc_prefix", "")
    if prefix and text.startswith(prefix):
        return text[len(prefix):]
    return text


def get_chroma_collection(chroma_path, backend):
    cfg = backend_config(backend)
    client = chromadb.PersistentClient(path=chroma_path)
    return client.get_collection(cfg["collection"]), cfg


def tokenize_bm25(text, stemmer=None):
    text = re.sub(r"[^\w\s]", "", text.lower())
    words = [w for w in text.split() if w and w not in STOPWORDS_ES]
    if stemmer is None:
        return words
    out = []
    for w in words:
        out.append(w if TECH_TOKEN_PATTERN.match(w) else stemmer.stem(w))
    return out


def build_bm25_index_from_lists(docs, metas):
    try:
        from nltk.stem.snowball import SnowballStemmer
        stemmer = SnowballStemmer("spanish")
    except Exception:
        stemmer = None
    tokenized = [tokenize_bm25(d, stemmer) for d in docs]
    return BM25Okapi(tokenized), docs, metas, stemmer


def build_bm25_index(collection):
    data = collection.get()
    docs, metas = data["documents"], data["metadatas"]
    return build_bm25_index_from_lists(docs, metas)


def dense_search(collection, embedder, query, k=100):
    q_vec = embedder.embed_one(query, is_query=True)
    res = collection.query(query_embeddings=[q_vec], n_results=k)
    return [
        {"text": doc, "meta": meta, "rank_dense": i}
        for i, (doc, meta) in enumerate(zip(res["documents"][0], res["metadatas"][0]))
    ]


def bm25_search(query, bm25_index, docs, metas, stemmer, k=100):
    scores = bm25_index.get_scores(tokenize_bm25(query, stemmer))
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [
        {"text": docs[i], "meta": metas[i], "rank_bm25": rank}
        for rank, i in enumerate(top_idx)
    ]


def rrf_fusion_baseline(dense, bm25, rrf_k=60, top_n=10):
    scores = {}
    for item in dense:
        cid = item["meta"]["chunk_id"]
        scores.setdefault(cid, {"text": item["text"], "meta": item["meta"], "score": 0.0})
        scores[cid]["score"] += 1.0 / (rrf_k + item["rank_dense"])
    for item in bm25:
        cid = item["meta"]["chunk_id"]
        scores.setdefault(cid, {"text": item["text"], "meta": item["meta"], "score": 0.0})
        scores[cid]["score"] += 1.0 / (rrf_k + item["rank_bm25"])
    return sorted(scores.values(), key=lambda x: x["score"], reverse=True)[:top_n]


collection, cfg = get_chroma_collection(CHROMA_PATH, EMBED_BACKEND)
embedder = Embedder(EMBED_BACKEND)
bm25_index, all_docs, all_metas, stemmer = build_bm25_index(collection)

# Opcional: restringir corpus a actas del batch (meta.sources)
def _doc_in_batch(doc_id: str, sources: set[str]) -> bool:
    if not sources:
        return True
    for src in sources:
        if doc_id == src or doc_id.startswith(src) or src in doc_id:
            return True
    return False


if batch_sources:
    idxs = [i for i, m in enumerate(all_metas) if _doc_in_batch(m.get("doc_id", ""), batch_sources)]
    if idxs:
        all_docs = [all_docs[i] for i in idxs]
        all_metas = [all_metas[i] for i in idxs]
        bm25_index, _, _, stemmer = build_bm25_index_from_lists(all_docs, all_metas)
        print(f"Corpus filtrado al batch: {len(all_docs)} chunks")
    else:
        print(f"⚠️  batch_sources no matcheó doc_id; usando corpus completo ({len(all_docs)} chunks)")
else:
    print(f"Corpus completo: {len(all_docs)} chunks")


def retrieve_baseline(query, top_n=TOP_N):
    d = dense_search(collection, embedder, query, k=RETRIEVAL_K)
    b = bm25_search(query, bm25_index, all_docs, all_metas, stemmer, k=RETRIEVAL_K)
    return rrf_fusion_baseline(d, b, rrf_k=RRF_K, top_n=top_n)


preview = retrieve_baseline("¿Qué se decidió sobre el talud?", top_n=3)
for h in preview:
    t = strip_doc_prefix(h["text"], EMBED_BACKEND)
    print(f"[{h['meta']['doc_id']}] {t[:80]}...")
print("✅ Baseline listo")


Corpus filtrado al batch: 168 chunks


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[254275-DO-AVO-15-V07] adas en los informes de revisión del PC y PGMA. AR-29 Estabilidad del talud || P...
[254275-DO-AVO-14-V07] envía en resultado de la verificación de la documentación aportada por la UTE so...
[254275-DO-AVO-16-V07] iará el certificado a la DF. AR-21 Plano camino provisional || DF solicita a la ...
✅ Baseline listo


## 5. Retrieval Cypher + puente a chunks

Tres modos de routing (parámetro `route`):

| `route` | Comportamiento |
|---|---|
| `factual` | Siempre Cypher local (1-hop + 2-hop) |
| `transversal` | Siempre list/agg |
| `hybrid` | Clasificador regex → elige factual o transversal |


In [17]:
_NORMALIZE_RE = re.compile(r"[^\wáéíóúñü]+", re.IGNORECASE)

TRANSVERSAL_TRIGGERS = [
    "todas", "todos", "todas las", "todos los", "agrupa", "agrupar", "clasifica",
    "frecuente", "frecuentes", "frecuencia", "lista", "listar", "enumera",
    "tipos de", "tipo de tareas", "patrón", "patrones", "conjunto", "varias actas",
    "todas las actas", "general", "globalmente", "qué incidencias", "qué problemas",
    "más comunes", "más frecuentes", "cuáles son las", "incumplimientos",
    "solicitudes realizadas",
]
FACTUAL_TRIGGERS = [
    "qué se decidió", "qué se acordó", "estado del", "estado de la", "estado de las",
    "cuál es", "cuáles es", "cuántos", "cuántas", "a qué altura", "modelo de",
    "función de", "sigla", "documentación", "certificados", "ar-29",
]
ENTITY_KEYWORDS = [
    "talud", "megafonía", "megafonia", "ute", "constructora", "luminaria", "andén",
    "anden", "ar-29", "hormigonado", "zapata", "camino provisional", "certificado",
    "hinca", "perfil", "deo", "df", "climatización", "climatizacion",
]


def _tokenize_match(s: str) -> set[str]:
    return set(t for t in _NORMALIZE_RE.split(s.lower()) if t and len(t) > 2 and t not in STOPWORDS_ES)


def _match_wordset(triple, doc, min_overlap=MATCH_MIN_OVERLAP):
    s, _, o = triple
    doc_words = _tokenize_match(strip_doc_prefix(doc, EMBED_BACKEND))
    if not doc_words:
        return False

    def overlap_ok(words):
        if not words:
            return False
        return len(words & doc_words) / len(words) >= min_overlap

    return overlap_ok(_tokenize_match(s)) and overlap_ok(_tokenize_match(o))


def chunks_matching_triple(triple, docs, metas):
    return [i for i, d in enumerate(docs) if _match_wordset(triple, d)]


def classify_query_regex(query: str) -> str | None:
    q = query.lower().strip()
    score_t = sum(1 for w in TRANSVERSAL_TRIGGERS if w in q)
    score_f = sum(1 for w in FACTUAL_TRIGGERS if w in q)
    if score_t >= 1 and score_t > score_f:
        return "transversal"
    if score_f >= 1 and score_f > score_t:
        return "factual"
    return None


def classify_query(query: str) -> str:
    label = classify_query_regex(query)
    return label if label else "factual"


def extract_seeds(query: str) -> list[str]:
    q = query.lower()
    seeds = [kw for kw in ENTITY_KEYWORDS if kw in q]
    if seeds:
        return seeds
    tokens = [t for t in _tokenize_match(query) if len(t) > 3]
    return tokens[:3]


def run_cypher(cypher: str, **params) -> tuple[list[dict], str]:
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(cypher, **params)
        rows = [dict(r) for r in result]
    return rows, cypher


CYPHER_FACTUAL = """
MATCH (e:Entity)-[r:RELATED]->(o:Entity)
WHERE ANY(seed IN $seeds WHERE e.norm CONTAINS seed OR o.norm CONTAINS seed)
RETURN e.name AS subject, r.predicate AS predicate, o.name AS object
LIMIT $limit
"""

CYPHER_FACTUAL_2HOP = """
MATCH (e:Entity)-[:RELATED]->(mid:Entity)-[r2:RELATED]->(o:Entity)
WHERE ANY(seed IN $seeds WHERE e.norm CONTAINS seed OR mid.norm CONTAINS seed)
RETURN e.name AS subject, r2.predicate AS predicate, o.name AS object
LIMIT $limit
"""

CYPHER_TRANSVERSAL_LIST = """
MATCH (a:Entity)-[r:RELATED]->(b:Entity)
WHERE ANY(kw IN $keywords WHERE r.predicate CONTAINS kw
       OR a.norm CONTAINS kw OR b.norm CONTAINS kw)
RETURN a.name AS subject, r.predicate AS predicate, b.name AS object
LIMIT $limit
"""

CYPHER_TRANSVERSAL_AGG = """
MATCH ()-[r:RELATED]->()
RETURN r.predicate AS predicate, count(*) AS cnt
ORDER BY cnt DESC
LIMIT $limit
"""


def rows_to_triples(rows: list[dict]) -> list[tuple]:
    triples = []
    for row in rows:
        if "subject" in row and "object" in row:
            triples.append((row["subject"], row.get("predicate", ""), row["object"]))
        elif "predicate" in row and "cnt" in row:
            triples.append((f"__agg__", row["predicate"], f"count={row['cnt']}"))
    return triples


def cypher_factual(query: str, limit: int = 50) -> tuple[list[tuple], list[dict], str]:
    seeds = extract_seeds(query)
    if not seeds:
        seeds = [norm_entity(w) for w in query.split()[:2]]
    rows1, cy1 = run_cypher(CYPHER_FACTUAL, seeds=seeds, limit=limit)
    rows2, cy2 = run_cypher(CYPHER_FACTUAL_2HOP, seeds=seeds, limit=limit)
    rows = rows1 + rows2
    cypher_used = cy1.strip() + "\n-- 2-hop --\n" + cy2.strip()
    return rows_to_triples(rows), rows, cypher_used


def cypher_transversal(query: str, limit: int = 100) -> tuple[list[tuple], list[dict], str]:
    q = query.lower()
    if any(w in q for w in ["frecuent", "más común", "más frecuente", "cuáles son las"]):
        rows, cy = run_cypher(CYPHER_TRANSVERSAL_AGG, limit=20)
    else:
        kws = extract_seeds(query) or ["incidencia", "solicitud", "problema", "acción", "accion"]
        rows, cy = run_cypher(CYPHER_TRANSVERSAL_LIST, keywords=kws, limit=limit)
    return rows_to_triples(rows), rows, cy


def chunks_from_triples(triples: list[tuple], docs, metas, max_chunks: int = TOP_N):
    seen, hits = set(), []
    for triple in triples:
        if triple[0] == "__agg__":
            continue
        for i in chunks_matching_triple(triple, docs, metas):
            cid = metas[i]["chunk_id"]
            if cid in seen:
                continue
            seen.add(cid)
            hits.append({"text": docs[i], "meta": metas[i], "score": 0.0, "from_graph": True})
            if len(hits) >= max_chunks:
                return hits
    return hits


def rrf_merge_lists(graph_hits, baseline_hits, rrf_k=RRF_K, top_n=TOP_N):
    scores = {}
    for rank, h in enumerate(graph_hits):
        cid = h["meta"]["chunk_id"]
        scores.setdefault(cid, {**h, "score": 0.0, "sources": set()})
        scores[cid]["score"] += 1.0 / (rrf_k + rank)
        scores[cid]["sources"].add("cypher")
    for rank, h in enumerate(baseline_hits):
        cid = h["meta"]["chunk_id"]
        scores.setdefault(cid, {**h, "score": 0.0, "sources": set()})
        scores[cid]["score"] += 1.0 / (rrf_k + rank)
        scores[cid]["sources"].add("baseline")
    out = sorted(scores.values(), key=lambda x: x["score"], reverse=True)[:top_n]
    for h in out:
        h.pop("sources", None)
    return out


def _resolve_route(query: str, route: str) -> str:
    """route: 'hybrid' | 'factual' | 'transversal'"""
    if route == "hybrid":
        return classify_query(query)
    if route in ("factual", "transversal"):
        return route
    raise ValueError(f"route desconocida: {route}")


def retrieve_cypher_rag(query: str, top_n: int = TOP_N, route: str = "hybrid"):
    """Retrieval Cypher + fallback baseline.

    route:
      - 'hybrid'       → clasificador regex (factual/transversal)
      - 'factual'      → siempre plantillas factual (1-hop + 2-hop)
      - 'transversal'  → siempre plantillas transversal (list/agg)
    """
    label = _resolve_route(query, route)
    debug = {"route_mode": route, "label": label, "cypher": "", "n_triples": 0, "n_graph_chunks": 0}
    if label == "factual":
        triples, _, cy = cypher_factual(query)
    else:
        triples, _, cy = cypher_transversal(query)
    debug["cypher"] = cy
    debug["n_triples"] = len(triples)
    graph_chunks = chunks_from_triples(triples, all_docs, all_metas, max_chunks=top_n)
    debug["n_graph_chunks"] = len(graph_chunks)
    baseline = retrieve_baseline(query, top_n=top_n)
    if len(graph_chunks) >= top_n // 2:
        merged = rrf_merge_lists(graph_chunks, baseline, top_n=top_n)
    else:
        merged = baseline
        debug["fallback_baseline"] = True
    return merged, triples, debug


def make_retrieve_fn(route: str):
    def _fn(q, k):
        chunks, _, _ = retrieve_cypher_rag(q, top_n=k, route=route)
        return chunks
    return _fn


print("✅ Cypher retrieval listo (rutas: factual | transversal | hybrid)")


✅ Cypher retrieval listo (rutas: factual | transversal | hybrid)


## 6. Generación + juez de respuesta


In [18]:
def build_prompt_baseline(query, chunks):
    blocks = []
    for i, ch in enumerate(chunks, 1):
        doc_id = ch["meta"]["doc_id"]
        text = strip_doc_prefix(ch["text"], EMBED_BACKEND)
        blocks.append(f"[Fragmento {i} - Fuente: {doc_id}]\n{text}")
    return f"""Eres COSORA, un asistente técnico especializado en análisis de actas de obra ferroviaria en España.

REGLAS:
1. Responde ÚNICAMENTE con información del CONTEXTO.
2. Si no está, dilo explícitamente.
3. Cita la fuente entre paréntesis: (Fuente: nombre_del_acta).

VOCABULARIO: DF=Dirección Facultativa, UTE=Unión Temporal de Empresas, DEO=Director de Ejecución de Obra.

=== CONTEXTO ===
{chr(10).join(blocks)}

=== PREGUNTA ===
{query}

=== RESPUESTA ==="""


def generate_answer(prompt: str, model: str = GEN_MODEL) -> str:
    if client is None:
        return "(OPENAI_API_KEY no configurada)"
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=800,
    )
    return (resp.choices[0].message.content or "").strip()


def retrieve_for_variant_full(query: str, variant: str, top_n: int = TOP_N):
    """variant: baseline | cypher_factual | cypher_transversal | cypher_hybrid"""
    if variant == "baseline":
        chunks = retrieve_baseline(query, top_n=top_n)
        return chunks, [], {}
    route = variant.replace("cypher_", "")
    return retrieve_cypher_rag(query, top_n=top_n, route=route)


def retrieve_for_variant(query: str, variant: str, top_n: int = TOP_N):
    chunks, _, _ = retrieve_for_variant_full(query, variant, top_n=top_n)
    return chunks


CITATION_RE = re.compile(r"\(Fuente:\s*([^)]+)\)", re.IGNORECASE)


def extract_cited_sources(answer: str) -> list[str]:
    return [m.strip() for m in CITATION_RE.findall(answer)]


def citations_grounded(cited: list[str], chunk_doc_ids: set[str]) -> bool:
    if not cited:
        return False
    for cite in cited:
        if not any(cite in did or did in cite for did in chunk_doc_ids if did):
            return False
    return True


def doc_overlap(ids_a: set[str], ids_b: set[str]) -> float:
    union = ids_a | ids_b
    return len(ids_a & ids_b) / len(union) if union else 1.0


def generate_answer_for_variant(query: str, variant: str, top_n: int = TOP_N):
    chunks, _, _ = retrieve_for_variant_full(query, variant, top_n=top_n)
    prompt = build_prompt_baseline(query, chunks)
    return generate_answer(prompt), chunks


def judge_answer(query: str, answer: str, context_blocks: list[str], model: str = LLM_JUDGE_MODEL) -> dict:
    """LLM-as-Judge sobre la RESPUESTA (graph_rag.ipynb sec. 9)."""
    ctx = "\n\n---\n\n".join(context_blocks[:8])
    system = (
        "Eres un evaluador estricto de respuestas de un asistente sobre actas de obra ferroviaria. "
        "Vas a puntuar la RESPUESTA en tres ejes (0/1/2). "
        'Responde EXACTAMENTE en formato JSON: {"correctness":X,"faithfulness":Y,"citation":Z}.'
    )
    user = (
        f"PREGUNTA:\n{query}\n\n"
        f"CONTEXTO PROVISTO AL ASISTENTE:\n{ctx}\n\n"
        f"RESPUESTA DEL ASISTENTE:\n{answer}\n\n"
        "Puntúa (0/1/2):\n"
        "- correctness: ¿la respuesta es correcta a la luz del contexto?\n"
        "- faithfulness: ¿se ciñe al contexto, sin inventar hechos no soportados?\n"
        "- citation: ¿cita fuentes (Fuente: ...) y las cita bien?\n"
        "Devuelve SOLO el JSON."
    )
    try:
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
            temperature=0,
            max_tokens=60,
        )
        raw = (resp.choices[0].message.content or "").strip()
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        if not m:
            return {"correctness": -1, "faithfulness": -1, "citation": -1, "raw": raw}
        data = json.loads(m.group(0))
        return {
            "correctness": int(data.get("correctness", -1)),
            "faithfulness": int(data.get("faithfulness", -1)),
            "citation": int(data.get("citation", -1)),
        }
    except Exception as e:
        return {"correctness": -1, "faithfulness": -1, "citation": -1, "error": str(e)}


def eval_answer_variant(variant: str, queries: list[str]):
    rows = []
    for q in queries:
        chunks, _, debug = retrieve_for_variant_full(q, variant)
        prompt = build_prompt_baseline(q, chunks)
        ans = generate_answer(prompt)
        ctx_blocks = [strip_doc_prefix(c["text"], EMBED_BACKEND) for c in chunks]
        scores = judge_answer(q, ans, ctx_blocks)
        cited = extract_cited_sources(ans)
        chunk_ids = {ch["meta"].get("doc_id", "") for ch in chunks}
        if variant != "baseline":
            base_chunks = retrieve_baseline(q, top_n=TOP_N)
            overlap = doc_overlap(chunk_ids, {c["meta"].get("doc_id", "") for c in base_chunks})
        else:
            overlap = 1.0
        rows.append({
            "query": q[:60],
            "variant": variant,
            "answer": ans,
            **scores,
            "has_citation": bool(cited),
            "n_citations": len(cited),
            "citations_grounded": citations_grounded(cited, chunk_ids),
            "n_triples": debug.get("n_triples", 0),
            "n_graph_chunks": debug.get("n_graph_chunks", 0),
            "fallback_baseline": debug.get("fallback_baseline", False),
            "doc_overlap_vs_baseline": overlap,
        })
    return rows


def summarize_answer_rows(rows: list[dict]) -> dict:
    def avg(field):
        vals = [r[field] for r in rows if r.get(field, -1) >= 0]
        return float(np.mean(vals)) if vals else 0.0
    variant = rows[0]["variant"] if rows else "?"
    c, f, ci = avg("correctness"), avg("faithfulness"), avg("citation")
    return {
        "variant": variant,
        "correctness": c,
        "faithfulness": f,
        "citation": ci,
        "overall": c * 0.5 + f * 0.35 + ci * 0.15,
        "pct_has_citation": float(np.mean([r["has_citation"] for r in rows])) if rows else 0.0,
        "pct_citations_grounded": float(np.mean([r["citations_grounded"] for r in rows])) if rows else 0.0,
        "avg_doc_overlap": float(np.mean([r["doc_overlap_vs_baseline"] for r in rows])) if rows else 0.0,
        "pct_fallback": float(np.mean([r["fallback_baseline"] for r in rows])) if rows else 0.0,
        "avg_n_triples": float(np.mean([r["n_triples"] for r in rows])) if rows else 0.0,
    }


# Demo rápida
demo_q = "¿Qué se decidió sobre el talud?"
for v in ("baseline", "cypher_hybrid"):
    ans, _ = generate_answer_for_variant(demo_q, v)
    print(f"[{v}] {ans[:200]}...")
print("✅ Generación + juez de respuesta listos")


[baseline] Se solicitó a la UTE que aportara información adicional para cerrar la verificación de la ejecución de hinca de perfiles en relación con la estabilidad del talud. Específicamente, se requería confirma...
[cypher_hybrid] Se decidió que para cerrar la verificación de la ejecución de hinca de perfiles en el talud, faltaba confirmar las profundidades reales de hinca (empotramiento en estratos competentes), separaciones y...
✅ Generación + juez de respuesta listos


## 7. Benchmark (21 queries)

Mismas 21 queries que `graph_rag.ipynb` (12 factuales + 9 transversales).

**4 variantes:** `baseline`, `cypher_factual`, `cypher_transversal`, `cypher_hybrid`.

### Calidad de respuesta (LLM-as-Judge)
correctness, faithfulness, citation → **overall** = 0.5·C + 0.35·F + 0.15·Ci

### Checks deterministas (sin juez LLM)
| Métrica | Qué mide |
|---|---|
| `pct_has_citation` | ¿Incluye `(Fuente: ...)`? |
| `pct_citations_grounded` | ¿Los doc_id citados están en los chunks recuperados? |
| `avg_doc_overlap` | Jaccard de doc_ids vs baseline (solo variantes Cypher) |
| `pct_fallback` | ¿Cayó al baseline por pocos graph_chunks? |
| `avg_n_triples` | Triples Cypher recuperados de media |


In [ ]:
BENCHMARK_QUERIES_FACTUAL = [
    "¿Qué se decidió sobre el talud?",
    "¿Cuál es el estado del camino provisional?",
    "¿Qué incidencias AR-29 aparecen?",
    "¿Qué responsable está asignado a las acciones sobre el talud?",
    "¿Qué se acordó sobre hormigonado de zapatas?",
    "Estado de las instalaciones de megafonía",
    "¿Qué documentación debe aportar la UTE sobre estabilidad?",
    "¿Cuál es la función de la sigla DEO en un acta de visita de obra?",
    "¿Cuántos días hay de plazo para devolver el acta firmada?",
    "¿Qué modelo de luminaria debe instalarse en los báculos del andén?",
    "¿Qué certificados de material debe aportar la UTE sobre las hincas de perfiles?",
    "¿A qué altura respecto al suelo se van a instalar las unidades exteriores de climatización?",
]

BENCHMARK_QUERIES_TRANSVERSAL = [
    "¿Cuáles son las incidencias más frecuentes en todas las actas?",
    "¿Qué elementos constructivos presentan más incidencias?",
    "¿Qué problemas pueden afectar a la seguridad de la explotación ferroviaria?",
    "Agrupa todas las incidencias detectadas en las actas y clasifícalas por tipo",
    "¿Cuáles son las acciones pendientes más frecuentes en las actas?",
    "¿Cuáles son los problemas más frecuentes en instalaciones eléctricas y luminarias?",
    "Lista todas las solicitudes realizadas a la constructora",
    "¿Qué incumplimientos normativos se detectan en el conjunto de actas?",
    "¿Qué tipo de tareas suelen quedar sin resolver?",
]

BENCHMARK_QUERIES = BENCHMARK_QUERIES_FACTUAL + BENCHMARK_QUERIES_TRANSVERSAL

ANSWER_VARIANTS = [
    "baseline",
    "cypher_factual",
    "cypher_transversal",
    "cypher_hybrid",
]

print(f"Benchmark: {len(BENCHMARK_QUERIES)} queries "
      f"({len(BENCHMARK_QUERIES_FACTUAL)} factual + {len(BENCHMARK_QUERIES_TRANSVERSAL)} transversal)")


def row_overall(r: dict) -> float:
    if r.get("correctness", -1) < 0:
        return 0.0
    return r["correctness"] * 0.5 + r["faithfulness"] * 0.35 + r["citation"] * 0.15


def summarize_deterministic(rows: list[dict]) -> dict:
    """Checks deterministas — calculados aquí para no depender de sec. 6 en memoria."""
    if not rows:
        return {"variant": "?"}
    return {
        "variant": rows[0]["variant"],
        "pct_has_citation": float(np.mean([bool(r.get("has_citation")) for r in rows])),
        "pct_citations_grounded": float(np.mean([bool(r.get("citations_grounded")) for r in rows])),
        "avg_doc_overlap": float(np.mean([r.get("doc_overlap_vs_baseline", 1.0) for r in rows])),
        "pct_fallback": float(np.mean([bool(r.get("fallback_baseline")) for r in rows])),
        "avg_n_triples": float(np.mean([r.get("n_triples", 0) for r in rows])),
    }


if ANSWER_EVAL_ENABLED and client is not None:
    all_rows = {}
    summaries = []

    for variant in ANSWER_VARIANTS:
        print(f"Evaluando · {variant}...")
        rows = eval_answer_variant(variant, BENCHMARK_QUERIES)
        all_rows[variant] = rows
        summaries.append(summarize_answer_rows(rows))

    sample = all_rows[ANSWER_VARIANTS[0]][0] if all_rows else {}
    if "has_citation" not in sample:
        print("⚠️  Re-ejecuta la celda sec. 6 antes del benchmark (faltan checks deterministas).")

    df_ans = pd.DataFrame(summaries).set_index("variant")
    llm_cols = [c for c in ("correctness", "faithfulness", "citation", "overall") if c in df_ans.columns]

    print("\n🏆 Calidad de RESPUESTA — LLM judge (global):")
    display(df_ans[llm_cols].sort_values("overall", ascending=False))

    df_det = pd.DataFrame([summarize_deterministic(all_rows[v]) for v in ANSWER_VARIANTS]).set_index("variant")
    print("\n📐 Checks deterministas (global):")
    display(df_det.sort_values("pct_citations_grounded", ascending=False))

    print("\n📊 Factuales (12) — overall:")
    rows_f = []
    for v in ANSWER_VARIANTS:
        sub = [all_rows[v][i] for i, q in enumerate(BENCHMARK_QUERIES) if q in BENCHMARK_QUERIES_FACTUAL]
        rows_f.append(summarize_answer_rows(sub))
    display(pd.DataFrame(rows_f).set_index("variant")[llm_cols].sort_values("overall", ascending=False))

    print("\n📊 Transversales (9) — overall:")
    rows_t = []
    for v in ANSWER_VARIANTS:
        sub = [all_rows[v][i] for i, q in enumerate(BENCHMARK_QUERIES) if q in BENCHMARK_QUERIES_TRANSVERSAL]
        rows_t.append(summarize_answer_rows(sub))
    display(pd.DataFrame(rows_t).set_index("variant")[llm_cols].sort_values("overall", ascending=False))

    print("\nΔ overall vs baseline (por query):")
    delta = []
    for i, q in enumerate(BENCHMARK_QUERIES):
        b = row_overall(all_rows["baseline"][i])
        row = {"query": q[:55], "baseline": b}
        for v in ("cypher_factual", "cypher_transversal", "cypher_hybrid"):
            row[v] = row_overall(all_rows[v][i])
            row[f"Δ_{v}"] = row[v] - b
        delta.append(row)
    display(pd.DataFrame(delta).sort_values("Δ_cypher_hybrid", ascending=False))

    print("\n⚠️ Queries sin cita o cita no grounded (cypher_hybrid):")
    issues = [
        r for r in all_rows["cypher_hybrid"]
        if not r.get("has_citation") or not r.get("citations_grounded")
    ]
    if issues:
        cols = [c for c in ("query", "has_citation", "citations_grounded", "n_triples", "fallback_baseline") if c in issues[0]]
        display(pd.DataFrame(issues)[cols])
    else:
        print("  Ninguna")
else:
    print("Benchmark omitido (ANSWER_EVAL_ENABLED=False o sin OPENAI_API_KEY)")


## 8. Pruebas iterativas por código

Edita `MIS_PREGUNTAS` y `VARIANTE` para inspeccionar el pipeline sin correr el benchmark completo.

`verbose=True` muestra Cypher, triples y chunks recuperados.


In [20]:
def ask_cypher_rag(query: str, variant: str = "cypher_hybrid", verbose: bool = True) -> str:
    """variant: baseline | cypher_factual | cypher_transversal | cypher_hybrid"""
    if variant == "baseline":
        chunks = retrieve_baseline(query, top_n=TOP_N)
        triples, debug = [], {}
    else:
        route = variant.replace("cypher_", "")
        chunks, triples, debug = retrieve_cypher_rag(query, top_n=TOP_N, route=route)

    if verbose:
        print(f"  variant={variant}")
        if debug:
            print(f"  route={debug.get('label')} | triples={debug.get('n_triples')} | graph_chunks={debug.get('n_graph_chunks')}")
            if debug.get("cypher"):
                print(f"  Cypher:\n    {debug['cypher'][:300]}...")
        print(f"  top chunks ({len(chunks)}):")
        for i, ch in enumerate(chunks[:3], 1):
            doc_id = ch["meta"].get("doc_id", "?")
            preview = strip_doc_prefix(ch["text"], EMBED_BACKEND)[:120].replace("\n", " ")
            print(f"    [{i}] {doc_id}: {preview}...")
        if triples:
            print(f"  sample triples ({min(3, len(triples))}):")
            for t in triples[:3]:
                if isinstance(t, dict):
                    s, p, o = t.get("subject", "?"), t.get("predicate", "?"), t.get("object", "?")
                else:
                    s, p, o = (list(t) + ["?", "?", "?"])[:3]
                print(f"    ({s}) -[{p}]-> ({o})")

    prompt = build_prompt_baseline(query, chunks)
    answer = generate_answer(prompt)
    return answer

In [21]:
# ─── Edita aquí ───────────────────────────────────────────────────────
VARIANTE = "cypher_hybrid"   # baseline | cypher_factual | cypher_transversal | cypher_hybrid

MIS_PREGUNTAS = [
    "¿Qué se decidió sobre el talud?",
    "¿Cuál es el estado del camino provisional?",
    "¿Qué incidencias aparecen relacionadas con instalaciones eléctricas?",
]

for q in MIS_PREGUNTAS:
    print("=" * 80)
    print(f"👤 {q}")
    print("-" * 80)
    resp = ask_cypher_rag(q, variant=VARIANTE, verbose=True)
    print(f"🤖 {resp}")


👤 ¿Qué se decidió sobre el talud?
--------------------------------------------------------------------------------
  variant=cypher_hybrid
  route=factual | triples=2 | graph_chunks=0
  Cypher:
    MATCH (e:Entity)-[r:RELATED]->(o:Entity)
WHERE ANY(seed IN $seeds WHERE e.norm CONTAINS seed OR o.norm CONTAINS seed)
RETURN e.name AS subject, r.predicate AS predicate, o.name AS object
LIMIT $limit
-- 2-hop --
MATCH (e:Entity)-[:RELATED]->(mid:Entity)-[r2:RELATED]->(o:Entity)
WHERE ANY(seed IN $se...
  top chunks (10):
    [1] 254275-DO-AVO-15-V07: adas en los informes de revisión del PC y PGMA. AR-29 Estabilidad del talud || Posterior a la reunión de obra, DF envía ...
    [2] 254275-DO-AVO-14-V07: envía en resultado de la verificación de la documentación aportada por la UTE sobre la estabilidad del talud, solicitand...
    [3] 254275-DO-AVO-16-V07: iará el certificado a la DF. AR-21 Plano camino provisional || DF solicita a la UTE que aporte la documentación gráfica ...
  sample triples 

In [22]:
# ─── Edita aquí ───────────────────────────────────────────────────────
VARIANTE = "cypher_transversal"   # baseline | cypher_factual | cypher_transversal | cypher_hybrid

MIS_PREGUNTAS = [
    "¿Qué se decidió sobre el talud?",
    "¿Cuál es el estado del camino provisional?",
    "¿Qué incidencias aparecen relacionadas con instalaciones eléctricas?",
]
for q in MIS_PREGUNTAS:
    print("=" * 80)
    print(f"👤 {q}")
    print("-" * 80)
    resp = ask_cypher_rag(q, variant=VARIANTE, verbose=True)
    print(f"🤖 {resp}")

👤 ¿Qué se decidió sobre el talud?
--------------------------------------------------------------------------------
  variant=cypher_transversal
  route=transversal | triples=2 | graph_chunks=0
  Cypher:
    
MATCH (a:Entity)-[r:RELATED]->(b:Entity)
WHERE ANY(kw IN $keywords WHERE r.predicate CONTAINS kw
       OR a.norm CONTAINS kw OR b.norm CONTAINS kw)
RETURN a.name AS subject, r.predicate AS predicate, b.name AS object
LIMIT $limit
...
  top chunks (10):
    [1] 254275-DO-AVO-15-V07: adas en los informes de revisión del PC y PGMA. AR-29 Estabilidad del talud || Posterior a la reunión de obra, DF envía ...
    [2] 254275-DO-AVO-14-V07: envía en resultado de la verificación de la documentación aportada por la UTE sobre la estabilidad del talud, solicitand...
    [3] 254275-DO-AVO-16-V07: iará el certificado a la DF. AR-21 Plano camino provisional || DF solicita a la UTE que aporte la documentación gráfica ...
  sample triples (2):
    (DF) -[envía]-> (resultado de la verificación de l